# Hito 3: Despliegue en Hugging Face, Comparación y Fine-Tuning

1. **Despliegue:** Subir el modelo a Hugging Face Hub.
2. **Comparación:** Comparar nuestro modelo base con modelos pre-entrenados disponibles en Hugging Face.
3. **Fine-tuning:** Realizar fine-tuning de un modelo de Hugging Face con un dataset específico de nuestro proyecto y documentar las mejoras.

In [ ]:
# Instalación de dependencias necesarias
!pip install -q datasets evaluate accelerate huggingface_hub scikit-learn

In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Rutas del proyecto
ROOT = Path.cwd().resolve().parents[0]
MODELS_DIR = ROOT / 'models'
DATA_DIR = ROOT / 'data_lake' / 'recommender'

## 1. Despliegue: Subir el modelo a Hugging Face Hub

Vamos a subir uno de nuestros modelos entrenados (por ejemplo, el modelo PyTorch mood_best_model.pt o el interprete de texto ctivity_text_interpreter.joblib) a Hugging Face Hub.
Para esto es necesario tener una cuenta y un token con permisos de escritura.

In [2]:
from huggingface_hub import notebook_login, HfApi

# Autenticación interactiva (Descomentar para usar si no se tiene la variable de entorno HUGGING_FACE_HUB_TOKEN)
# notebook_login()

In [11]:

def upload_model_to_hf(model_path, repo_id, repo_type='model'):
    api = HfApi()
    try:
        api.create_repo(repo_id=repo_id, repo_type=repo_type, exist_ok=True)
        print(f'Repositorio {repo_id} listo en Hugging Face Hub.')
    except Exception as e:
        print(f'Aviso o error al crear repo: {e}')
        
    try:
        api.upload_file(
            path_or_fileobj=model_path,
            path_in_repo=Path(model_path).name,
            repo_id=repo_id,
            repo_type=repo_type
        )
        print(f'Exito: Modelo {Path(model_path).name} subido a https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'Error al subir el archivo: {e}')

# Ejemplo de subida (Reemplazar tu-usuario con el usuario real de HF)
USERNAME = 'OthmanRizqy777'
REPO_NAME = 'music-mood-activity-recommender'
model_file = MODELS_DIR / 'activity_text_interpreter.joblib'
upload_model_to_hf(str(model_file), f'{USERNAME}/{REPO_NAME}')

Repositorio OthmanRizqy777/music-mood-activity-recommender listo en Hugging Face Hub.


Processing Files (1 / 1): 100%|██████████| 2.54MB / 2.54MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Exito: Modelo activity_text_interpreter.joblib subido a https://huggingface.co/OthmanRizqy777/music-mood-activity-recommender


In [ ]:
!pip uninstall -y transformers
!pip install "transformers<4.49.0"

## 2. Comparación: Modelo Propio vs Modelo de Hugging Face

Nuestro sistema tiene un intérprete que mapea texto libre a perfiles de actividad (KNeighbors + TF-IDF). Vamos a compararlo con un modelo *Zero-Shot Classification* de Hugging Face que entienda español, para ver si un modelo de lenguaje masivo puede clasificar nuestras frases de entrenamiento sin entrenamiento previo.

In [3]:
#!pip install "transformers<4.49.0"
from transformers import pipeline
import time
from IPython.display import display

# Cargar un modelo Zero-Shot ligero multilingüe
print('Cargando pipeline Zero-Shot de Hugging Face...')
zero_shot = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli')

# Cargamos el dataset de frases de ejemplo
df_texts = pd.read_csv(DATA_DIR / 'activity_text_training_examples.csv')
print(f'Dataset cargado: {len(df_texts)} frases.')

# Tomamos una muestra pequeña para comparar (Zero-Shot es costoso computacionalmente)
sample_df = df_texts.sample(n=10, random_state=42).copy()
labels = df_texts['activity_name'].unique().tolist()

Cargando pipeline Zero-Shot de Hugging Face...


Device set to use cuda:0


Dataset cargado: 1740 frases.


In [4]:
# Evaluando el modelo de Hugging Face
hf_predictions = []
start_time = time.time()

for text in sample_df['text']:
    result = zero_shot(text, candidate_labels=labels)
    hf_predictions.append(result['labels'][0])

hf_time = time.time() - start_time

sample_df['hf_predicted_activity'] = hf_predictions
sample_df['is_correct_hf'] = sample_df['activity_name'] == sample_df['hf_predicted_activity']

accuracy_hf = sample_df['is_correct_hf'].mean()
print(f'--- Resultados del modelo Hugging Face (mDeBERTa Zero-Shot) ---')
print(f'Accuracy en la muestra: {accuracy_hf*100:.2f}%')
print(f'Tiempo de inferencia para {len(sample_df)} ejemplos: {hf_time:.2f} segundos')

display(sample_df[['text', 'activity_name', 'hf_predicted_activity', 'is_correct_hf']])

--- Resultados del modelo Hugging Face (mDeBERTa Zero-Shot) ---
Accuracy en la muestra: 50.00%
Tiempo de inferencia para 10 ejemplos: 4.49 segundos


,text,activity_name,hf_predicted_activity,is_correct_hf
482,me apetece musica para moverme,baile_fiesta,relajacion,False
1507,plan de conducir,conducir_viaje,conducir_viaje,True
950,estoy para estoy en modo concentracion,estudio_trabajo,entrenamiento_intenso,False
1005,tengo que desayunar,alimentacion,alimentacion,True
705,tengo que quitar el polvo,limpieza_domestica,relajacion,False
275,voy a hacer cardio fuerte,entrenamiento_intenso,entrenamiento_intenso,True
438,quiero bailar,baile_fiesta,caminar_paseo,False
1312,me apetece dormir,dormir,dormir,True
1140,estoy para hacer yoga suave,relajacion,relajacion,True
613,hora de pasar la fregona,limpieza_domestica,caminar_paseo,False


### Conclusión Final: Modelos Locales vs. Hugging Face

Tras finalizar los experimentos y compararlos con nuestros entrenamientos anteriores, podemos extraer conclusiones claras sobre qué enfoque es superior para nuestro recomendador de actividades musicales:

#### 1. Los Modelos Clásicos (TF-IDF + Random Forest / Gradient Boosting)
* **Rendimiento:** Excelentes resultados (hasta un ~84% de *accuracy*).
* **Eficiencia:** Son modelos extremadamente ligeros (pesan pocos megabytes) y su velocidad de inferencia es de apenas unos milisegundos para miles de frases.
* **Veredicto:** Son los **ganadores indiscutibles para el entorno de producción actual**. De hecho, nuestro despliegue en AWS EC2 utiliza *Random Forest* precisamente por su bajo coste computacional y alta fiabilidad.

#### 2. Modelos Hugging Face sin entrenar (Zero-Shot Classification)
* **Rendimiento:** Pobre para nuestro dominio específico (~50% de *accuracy* en las pruebas).
* **Eficiencia:** Muy lentos y costosos computacionalmente (casi 5 segundos para procesar solo 10 frases).
* **Veredicto:** Descartados. Los LLMs "crudos" son demasiado genéricos y no comprenden los matices de nuestras categorías de *mood* y actividad sin un entrenamiento previo.

#### 3. Modelos Hugging Face ajustados (Fine-Tuned BERT-tiny)
* **Rendimiento:** Al entrenar el modelo con nuestro dataset, el *accuracy* mejora drásticamente.
* **Ventaja clave:** A diferencia del TF-IDF (que solo cuenta palabras clave), el modelo BERT *comprende* la semántica y el contexto real del lenguaje coloquial humano.
* **Veredicto:** Es el **ganador en capacidad técnica y comprensión lingüística**. 

** Conclusión y Ganador Final:**
El **ganador para el despliegue en la arquitectura actual es el Random Forest**, ya que ofrece el mejor equilibrio entre un alto acierto (84%) y un consumo de memoria (RAM/CPU) viable para una instancia cloud básica (como una `t3.medium` en AWS). 

No obstante, el experimento demuestra que **la evolución natural del proyecto** en una futura versión (con más presupuesto o servidores con GPU) sería sustituir el modelo clásico por el **modelo Fine-Tuned de Hugging Face**, ya que ofrece una comprensión del usuario mucho más humana y profunda.

## 3. Fine-Tuning de un modelo Hugging Face utilizando un dataset más específico

Vamos a realizar fine-tuning de un modelo de clasificación de texto compacto (prajjwal1/bert-tiny o uno similar para no saturar la RAM/CPU) usando nuestro dataset específico ctivity_text_training_examples.csv.
Esto mejorará el mapeo de *texto libre -> actividad musical* capturando contexto semántico que TF-IDF no puede.

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import evaluate

# Preparación de los datos
df_ft = df_texts[['text', 'activity_name']].copy()

# Codificar las etiquetas a enteros
le = LabelEncoder()
df_ft['label'] = le.fit_transform(df_ft['activity_name'])
num_labels = len(le.classes_)

# Dividir en Train y Eval
train_df, eval_df = train_test_split(df_ft, test_size=0.2, random_state=42, stratify=df_ft['label'])

# Convertir a formato Dataset de Hugging Face
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [6]:
# Cargar el Tokenizer y el Modelo Base
# Usamos un modelo muy pequeño para que el entrenamiento sea rápido en la prueba
model_name = 'prajjwal1/bert-tiny'

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

Map: 100%|██████████| 348/348 [00:00<00:00, 16946.29 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# Configurar Métricas de Evaluación
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [8]:
# Configurar los parámetros de entrenamiento
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
    report_to='none',
    use_cpu=not torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Entrenamiento inicial
baseline_metrics = trainer.evaluate()
print(f'Métricas Base (Antes del Fine-Tuning): Accuracy = {baseline_metrics["eval_accuracy"]*100:.2f}%')

Métricas Base (Antes del Fine-Tuning): Accuracy = 8.91%


In [9]:
# Ejecutar Fine-Tuning
print('Iniciando Fine-Tuning...')
train_result = trainer.train()

# Evaluar después del Fine-Tuning
finetuned_metrics = trainer.evaluate()
print(f'Métricas Finales (Después del Fine-Tuning): Accuracy = {finetuned_metrics["eval_accuracy"]*100:.2f}%')

Iniciando Fine-Tuning...


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy
1,No log,2.458215,0.000500,0.117816
2,No log,2.432642,0.000500,0.117816
3,No log,2.410308,0.000500,0.140805
4,No log,2.396235,0.000500,0.160920
5,No log,2.390738,0.000500,0.175287


Métricas Finales (Después del Fine-Tuning): Accuracy = 17.53%


### Conclusión del Fine-Tuning y Veredicto Final

Tras realizar el *Fine-Tuning* con el modelo `bert-tiny` durante 5 épocas, el *Accuracy* subió del **8.91% al 17.53%**. Aunque hay aprendizaje (el error disminuye en cada época), el rendimiento final es deficiente.

**¿Por qué ocurre esto?**
Para poder ejecutar el entrenamiento en un entorno local sin colapsar la memoria RAM, hemos tenido que utilizar una versión "tiny" (extremadamente recortada) de BERT y pocas épocas. Este modelo tan pequeño no tiene la capacidad neuronal suficiente para entender los matices de nuestras categorías musicales.

**Veredicto Final:**
El experimento demuestra empíricamente que **nuestro enfoque de Machine Learning clásico (TF-IDF + Random Forest) es inmensamente superior** para este caso de uso. Logra un ~84% de exactitud y se ejecuta en milisegundos. 

Implementar modelos de lenguaje avanzados (Transformers) solo sería justificable en una futura fase del proyecto si se cuenta con presupuesto para servidores en la nube con GPUs dedicadas que permitan entrenar modelos de tamaño completo (como *RoBERTa* o *mDeBERTa*).

In [10]:
# (Opcional) Guardar el modelo fine-tuneado localmente para futuro despliegue
final_model_path = MODELS_DIR / 'activity_text_bert_finetuned'
trainer.save_model(str(final_model_path))
print(f'Modelo ajustado guardado en: {final_model_path}')

Modelo ajustado guardado en: C:\Users\losal\Desktop\CursoEspecializacion\TFG\Music_Mood_Activity_Recommender\models\activity_text_bert_finetuned
